# Haas Japanese Colab Notebook

މި ނޯޓްބުކުން Third Intelligence **Haas-ja-v1.0** ބޭނުންކޮށް ޖަޕާނީ އަޑާއި ވީޑިއޯ ފައިލްތައް އަވަހަށް ލިޔުމަށް ބަދަލުކުރެއެވެ.

## ޚާއްސިއްޔަތުތައް

- **ހިލޭ:** Google Colab ގެ ހިލޭ GPU runtime ގައި ދުއްވައެވެ.
- **ޖަޕާނީ އަޑު ދެނެގަންނަ މޮޑެލް:** Third Intelligence `third-intelligence/Haas-ja-v1.0` ބޭނުންކުރެއެވެ؛ މިއީ ޖަޕާނީ އަޑު ޓެކްސްޓަށް ބަދަލުކުރާ ހަމައެކަނި 0.6B-parameter ގެ މޮޑެލެކެވެ.
- **ނިޝާންތައް:** މޮޑެލުން `、` އަދި `。` ފަދަ ޖަޕާނީ punctuation ލަފާކޮށް ނެރެއެވެ.
- **ވަގުތު ނިޝާންތަކާއި ސަބްޓައިޓަލް:** `subtitle` creates SRT with approximate chunk boundaries, not native word/character timestamps; `none` disables timing.
- **ދިގު ފައިލްތައް:** ދިގު އަޑު/ވީޑިއޯ ފައިލްތައް ތެރޭގައި ކުޑަ audio chunk ތަކަށް ބައިކޮށް ތަރުތީބުން transcript ކުރެއެވެ.
- **Google Drive:** ފައިލްތައް ސީދާ upload ކުރަން ނުވަތަ Google Drive ގައި ހުންނަ ފައިލްތައް process ކުރަން ހެދޭނެއެވެ.
- **ގިނަ output format:** TXT / Markdown / JSON / CSV / XLSX / SRT އުފައްދައި ZIP archive އެއްގައި ބަންދުކުރެއެވެ.

## ބޭނުންކުރާނެ ގޮތް

1. **1. Settings** ގައި `INPUT_MODE` ހޮވައި، ބޭނުންވާނަމަ އެހެން settings ބަދަލުކުރާށެވެ.
2. **Runtime → Run all** ހޮވާށެވެ.
3. `INPUT_MODE = "upload"` ނަމަ، **2. File selection / Google Drive mount** ގައި ފައިލްތައް ހޮވާށެވެ.
4. ޕްރޮސެސް ނިމުމުން ނަތީޖާގެ ZIP އޮޓޮމެޓިކްކޮށް download ވާނެއެވެ.

## ނޯޓްތައް

- GPU ނުލިބެންޏާ ނޯޓްބުކުން transcription ފަށާކަށް ކުރިން ހުއްޓާލައެވެ.
- ޕްރޮސެސިންގ ވަގުތު ބަދަލުވެއެވެ. ފުރަތަމަ run ގައި transcription ފަށާކަށް ކުރިން Colab ގައި NeMo އާއި ހަމައެކަނި 1.23 GB ގެ މޮޑެލް download ކުރަން ޖެހެއެވެ. ސީދާ upload އަށްވެސް upload ވަގުތު ބޭނުންވެއެވެ.
- ދިގު އަޑު internal chunk ތަކުގައި process ކުރާތީ chunk boundary ގެ ކައިރީގައި context ކެނޑި recognition އަށް އަސަރުކުރެވިދާނެއެވެ. `CHUNK_SECONDS` ބޮޑުކުރުމުން boundary މަދުވާ ނަމަވެސް ކޮންމެ inference އެއް ބުރަވެއެވެ؛ ކުޑަކުރުމުން inference ފަސޭހަވާ ނަމަވެސް boundary ގިނަވެއެވެ.
- Third Intelligence މޮޑެލް **Apache-2.0** ލައިސަންސް ގެ ދަށުން ހުރެއެވެ.


In [ ]:
# @title 1. ސެޓިންގްސް
# ============================================================
# 1. ސެޓިންގްސް
# ============================================================
import os
os.environ["USE_TF"] = "0"  # Set before importing Transformers.
from pathlib import Path

# @markdown ## އަޑު / ވީޑިއޯ ފައިލްތައް
# @markdown - `INPUT_MODE`: ފައިލްތައް ސީދާ upload ކުރުމަށް `upload` ބޭނުންކުރާށެވެ، ނުވަތަ Google Drive ގައި ކުރިން ސޭވްކުރެވިފައިވާ ފައިލްތައް ކިޔުމަށް `google_drive` ބޭނުންކުރާށެވެ.
INPUT_MODE = "upload"  # @param ["upload", "google_drive"]

# @markdown ### Google Drive އިން ފައިލްތައް ކިޔާ
# @markdown މި settings ތައް ބޭނުންކުރަނީ `INPUT_MODE == "google_drive"` ނަމަ އެކަންޏެވެ.
# @markdown - `GOOGLE_DRIVE_ROOT`: އާންމުކޮށް `/content/drive/MyDrive` ގޮތަށް ބަހައްޓާށެވެ.
# @markdown - `GOOGLE_DRIVE_SUBPATH`: MyDrive ގެ ދަށުގައި ހުންނަ ފައިލެއް ނުވަތަ ފޯލްޑަރެއްގެ path. MyDrive ގެ މަތީ ފޯލްޑަރ ބޭނުންކުރުމަށް ހުސްކޮށް ބަހައްޓާށެވެ.
# @markdown - `RECURSIVE`: Drive input path އަކީ folder އެއް ނަމަ subfolder ތައްވެސް ހޯދާށެވެ.
# @markdown - `FILENAME_CONTAINS`: އިޚްތިޔާރީ filename substring filter.
GOOGLE_DRIVE_ROOT = "/content/drive/MyDrive"  # @param {type:"string"}
GOOGLE_DRIVE_SUBPATH = ""  # @param {type:"string"}
RECURSIVE = False  # @param {type:"boolean"}
FILENAME_CONTAINS = ""  # @param {type:"string"}

# @markdown ## Decoder
# @markdown - Haas-ja-v1.0 uses TDT only; CTC is unavailable.
MODEL_ID = "third-intelligence/Haas-ja-v1.0"

# @markdown ## Timestamps
# @markdown - `TIMESTAMP_MODE`: `none` adds no timing information. `subtitle` creates SRT using approximate chunk boundaries, not native word timestamps.
TIMESTAMP_MODE = "subtitle"  # @param ["subtitle", "none"]

# @markdown ## ދިގު އަޑު
# @markdown - `CHUNK_SECONDS`: ދިގު ފައިލެއް ކޮންމެ internal chunk އެއްގައި ކިހާ ސިކުންތު ޕްރޮސެސްކުރާނެކަން ކޮންޓްރޯލްކުރެއެވެ. ބޮޑު value ތަކުން chunk boundary މަދުކޮށް context ގިނަކޮށް ބަހައްޓާ ނަމަވެސް ކޮންމެ inference އެއް ބުރަވެ GPU memory ގިނައިން ބޭނުންކުރެވިދާނެއެވެ. ކުޑަ value ތަކުން inference ފަސޭހަވާ ނަމަވެސް boundary ގިނަވެ އެ ތަންތަނުގެ ކައިރީގައި recognition ކެނޑިދާނެއެވެ. އާންމުކޮށް default value ބަދަލުނުކޮށް ބަހައްޓާށެވެ. އަސްލު input file އަށް ބަދަލެއް ނުގެނެވެއެވެ.
CHUNK_SECONDS = 28.0  # @param {type:"number"}
INFERENCE_BATCH_SIZE = 1

# ސަބްޓައިޓަލް ދައްކާ ގޮތް (ތެރޭގެ settings)
SUBTITLE_MAX_LINE_CHARS = 20

# @markdown ## Output
# @markdown - `SAVE_OUTPUTS_TO_GOOGLE_DRIVE`: `True` ނަމަ input method އަކަށް ނުބަލައި output ZIP އެ Google Drive ގައި ސޭވްކުރާށެވެ.
# @markdown - `AUTO_DOWNLOAD_ZIP`: ނިމުމުގައި ZIP archive އެއް ހަދައި download ކުރާށެވެ.
# - Output format ތައް ކޮންޓްރޯލްކުރަނީ ތިރީގައިވާ Boolean flag ތަކުންނެވެ.
SAVE_OUTPUTS_TO_GOOGLE_DRIVE = False  # @param {type:"boolean"}
AUTO_DOWNLOAD_ZIP = True  # @param {type:"boolean"}
SAVE_TXT = True
SAVE_MD = True
SAVE_JSON = True
SAVE_CSV = True
SAVE_XLSX = True
SAVE_SRT = True
INCLUDE_TIMESTAMPS_IN_TXT = True

# Advanced
REQUIRE_CUDA = True
UPLOAD_DIR = "/content/haas_ja_uploads"
WORK_DIR = "/content/haas_ja_work"
OUTPUT_DIR = "/content/haas_ja_outputs"
ZIP_FILE_NAME = "haas_ja_outputs.zip"

SUPPORTED_EXTS = {
    ".aac", ".avi", ".flac", ".m4a", ".m4v", ".mkv", ".mov",
    ".mp3", ".mp4", ".ogg", ".opus", ".wav", ".webm", ".wma"
}

RETURN_TIMESTAMPS = TIMESTAMP_MODE == "subtitle"

if TIMESTAMP_MODE not in {"subtitle", "none"}:
    raise ValueError("TIMESTAMP_MODE must be subtitle / none.")
if not 0 < CHUNK_SECONDS <= 28:
    raise ValueError("CHUNK_SECONDS must be between 0 and 28.")

for p in [UPLOAD_DIR, WORK_DIR, OUTPUT_DIR]:
    Path(p).mkdir(parents=True, exist_ok=True)

print("Settings ލޯޑްކުރެވިއްޖެ.")


In [ ]:
# @title 2. ފައިލް ހޮވުން / Google Drive mount
# ============================================================
# 2. ފައިލް ހޮވުން / Google Drive mount
# ============================================================
from pathlib import Path
import shutil

def collect_supported_files(root: Path):
    if root.is_file():
        files = [root]
    elif root.is_dir():
        pattern = "**/*" if RECURSIVE else "*"
        files = [p for p in root.glob(pattern) if p.is_file()]
    else:
        raise FileNotFoundError(f"Input path ނެތް: {root}")
    files = [p for p in files if p.suffix.lower() in SUPPORTED_EXTS]
    if FILENAME_CONTAINS:
        files = [p for p in files if FILENAME_CONTAINS in p.name]
    return sorted(files)

selected_files = []

if INPUT_MODE == "upload":
    from google.colab import files
    uploaded = files.upload()
    upload_dir = Path(UPLOAD_DIR)
    for name, data in uploaded.items():
        target = upload_dir / Path(name).name
        target.write_bytes(data)
        if target.suffix.lower() in SUPPORTED_EXTS:
            selected_files.append(target)
elif INPUT_MODE == "google_drive":
    from google.colab import drive
    drive.mount("/content/drive")
    root = Path(GOOGLE_DRIVE_ROOT)
    if GOOGLE_DRIVE_SUBPATH:
        root = root / GOOGLE_DRIVE_SUBPATH
    selected_files = collect_supported_files(root)
else:
    raise ValueError("INPUT_MODE އަކީ upload ނުވަތަ google_drive ވާންޖެހެއެވެ.")

if not selected_files:
    raise RuntimeError("ސަޕޯޓް ކުރާ audio/video ފައިލެއް ހޮވާފައެއް ނެތް.")

print("ހޮވިފައިވާ ފައިލްތައް:")
for p in selected_files:
    print(" -", p)


In [ ]:
# @title 3. GPU ޗެކް އަދި dependency installation
# ============================================================
# 3. GPU ޗެކް އަދި dependency installation
# ============================================================
import os, shutil, subprocess, sys
from pathlib import Path

if os.environ.get("USE_TF") != "0":
    raise RuntimeError("Run section 1 first in a fresh runtime (USE_TF=0 must be set before Transformers imports).")

import torch

if REQUIRE_CUDA and not torch.cuda.is_available():
    raise RuntimeError("GPU ނުލިބެއެވެ. Colab ގައި “Runtime → Change runtime type → GPU” ހޮވާށެވެ.")

print("Torch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("GPU compute capability:", torch.cuda.get_device_capability(0))
print("BF16 ސަޕޯޓް:", torch.cuda.is_bf16_supported())

if shutil.which("ffmpeg") is None or shutil.which("ffprobe") is None:
    subprocess.check_call(["apt-get", "-qq", "update"])
    subprocess.check_call(["apt-get", "-qq", "install", "-y", "ffmpeg"])

# Official Haas-ASR source pinned to a reviewed upstream commit.
# Keep the installer and the preflight on the same Transformers version.
TRANSFORMERS_VERSION = "4.57.6"
install_command = [
    sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--progress-bar", "off",
    "haas-asr @ git+https://github.com/third-intelligence/Haas-ASR.git@acc614f68769dcbfae9dba96b48439c4a66295af",
    f"transformers=={TRANSFORMERS_VERSION}", "pandas", "openpyxl",
]
# Preserve the real pip error rather than a quiet CalledProcessError.
installation = subprocess.run(
    install_command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
log_path = Path("/content/haas_ja_pip_install.log")
log_path.write_text(installation.stdout or "", encoding="utf-8")
if installation.returncode != 0:
    print(f"pip install failed (exit {installation.returncode}). Full log: {log_path}")
    print("\n".join((installation.stdout or "").splitlines()[-120:]))
    raise RuntimeError(f"Dependency installation failed; full pip log: {log_path}")
print("Dependency installation ނިމިއްޖެ.")
print("Full pip log:", log_path)

# Check the decoder in a fresh Python interpreter before downloading the model.
preflight = (
    "import importlib.metadata, os, sys; "
    "assert os.environ.get('USE_TF') == '0', 'USE_TF must be 0'; "
    f"assert importlib.metadata.version('transformers') == {TRANSFORMERS_VERSION!r}; "
    "from transformers.utils import is_tf_available; "
    "assert not is_tf_available(), 'TensorFlow import must be disabled'; "
    "import haas_asr; from haas_asr._fla import check_fla_version; check_fla_version(); "
    "from fla.layers.kda import KimiDeltaAttention; "
    "assert 'tensorflow' not in sys.modules, 'TensorFlow loaded unexpectedly'"
)
subprocess.check_call([sys.executable, "-c", preflight])
print("Decoder import check passed. Continue to model loading.")


In [ ]:
# @title 4. އެހީތެރި function ތައް
# ============================================================
# 4. އެހީތެރި function ތައް
# ============================================================
from pathlib import Path
import csv, json, math, re, shutil, subprocess
import pandas as pd

ASCII_WORD = re.compile(r"[A-Za-z0-9]")

def safe_stem(path: Path) -> str:
    name = re.sub(r'[\\/:*?"<>|]+', "_", path.stem).strip()
    return name or "transcription"

def run_command(cmd):
    cp = subprocess.run(cmd, capture_output=True, text=True)
    if cp.returncode != 0:
        raise RuntimeError((cp.stderr or cp.stdout or "Command ދުއްވުން ފޭލްވެއްޖެ.").strip())
    return cp

def convert_to_mono_16k_wav(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    run_command([
        "ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
        "-i", str(src), "-vn", "-ac", "1", "-ar", "16000",
        "-c:a", "pcm_s16le", str(dst)
    ])
    if not dst.exists() or dst.stat().st_size == 0:
        raise RuntimeError(f"Audio ނެރުން ފޭލްވެއްޖެ: {src}")

def join_text_parts(parts):
    result = ""
    for part in parts:
        part = (part or "").strip()
        if not part:
            continue
        if result and ASCII_WORD.search(result[-1]) and ASCII_WORD.search(part[0]):
            result += " "
        result += part
    return result

def estimated_segment(text: str, start: float, end: float, chunk_index: int):
    """Approximate subtitle span: chunk boundaries, not forced alignment."""
    return {
        "type": "segment", "text": text, "start": start, "end": end,
        "duration": end - start, "chunk_index": chunk_index,
        "timing_source": "approximate_chunk_bounds",
    }

def build_subtitle_cues(all_ts):
    if TIMESTAMP_MODE != "subtitle":
        return [], "none"
    cues = [
        {"start": x["start"], "end": x["end"], "text": x["text"]}
        for x in all_ts["segment"] if x["text"].strip()
    ]
    return cues, "approximate_chunk_bounds"

def format_srt_timestamp(seconds):
    ms = max(0, int(round(float(seconds) * 1000)))
    h, rem = divmod(ms, 3600000)
    m, rem = divmod(rem, 60000)
    s, ms = divmod(rem, 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

def wrap_cue_text(text, max_line_chars):
    text = (text or "").strip()
    if not text or len(text) <= max_line_chars:
        return text
    target = len(text) // 2
    candidates = [i + 1 for i, ch in enumerate(text) if ch in "、。！？!?，,．."]
    candidates = [i for i in candidates if 1 <= i < len(text)]
    split = min(candidates, key=lambda i: abs(i - target)) if candidates else min(max_line_chars, len(text)-1)
    return text[:split].rstrip() + "\n" + text[split:].lstrip()

def cues_to_srt(cues):
    blocks = []
    for i, cue in enumerate(cues, 1):
        blocks.append(
            f"{i}\n"
            f"{format_srt_timestamp(cue['start'])} --> {format_srt_timestamp(cue['end'])}\n"
            f"{wrap_cue_text(cue['text'], SUBTITLE_MAX_LINE_CHARS)}"
        )
    return "\n\n".join(blocks) + ("\n" if blocks else "")

def flatten_timestamps(all_ts):
    rows = []
    for kind in ("segment", "word", "char"):
        rows.extend(all_ts[kind])
    return rows

def write_outputs(source_path, payload, out_dir):
    stem = safe_stem(source_path)
    paths = []
    text = payload["text"]
    cues = payload["subtitle_cues"]

    if SAVE_TXT:
        p = out_dir / f"{stem}.txt"
        parts = [text]
        if INCLUDE_TIMESTAMPS_IN_TXT and cues:
            parts += ["", "=== Timestamps ==="]
            parts += [
                f"[{format_srt_timestamp(c['start'])} --> {format_srt_timestamp(c['end'])}] {c['text']}"
                for c in cues
            ]
        p.write_text("\n".join(parts) + "\n", encoding="utf-8")
        paths.append(p)

    if SAVE_MD:
        p = out_dir / f"{stem}.md"
        body = f"# {source_path.name}\n\n{text}\n"
        if cues:
            body += "\n## Timestamps\n\n"
            body += "\n".join(
                f"- `{format_srt_timestamp(c['start'])} --> {format_srt_timestamp(c['end'])}` {c['text']}"
                for c in cues
            )
            body += "\n"
        p.write_text(body, encoding="utf-8")
        paths.append(p)

    if SAVE_JSON:
        p = out_dir / f"{stem}.json"
        p.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
        paths.append(p)

    rows = flatten_timestamps(payload["timestamps"])
    if SAVE_CSV:
        p = out_dir / f"{stem}.csv"
        pd.DataFrame(rows, columns=["type","text","start","end","duration","chunk_index"]).to_csv(
            p, index=False, encoding="utf-8-sig"
        )
        paths.append(p)

    if SAVE_XLSX:
        p = out_dir / f"{stem}.xlsx"
        with pd.ExcelWriter(p, engine="openpyxl") as writer:
            for kind in ("segment", "word", "char"):
                pd.DataFrame(payload["timestamps"][kind]).to_excel(
                    writer, sheet_name=kind + "s", index=False
                )
            pd.DataFrame(payload["chunks"]).to_excel(writer, sheet_name="chunks", index=False)
            pd.DataFrame(cues).to_excel(writer, sheet_name="subtitles", index=False)
        paths.append(p)

    if SAVE_SRT and TIMESTAMP_MODE == "subtitle" and cues:
        p = out_dir / f"{stem}.srt"
        p.write_text(cues_to_srt(cues), encoding="utf-8")
        paths.append(p)

    return paths


In [ ]:
# @title 5. Haas Japanese ލޯޑްކުރުން
# ============================================================
# 5. Haas Japanese ލޯޑްކުރުން
# ============================================================
import torch
import haas_asr

torch.set_float32_matmul_precision("high")
# Use CUDA compute capability: T4 (SM 7.5) has no native BF16 even if a support flag says True.
model_dtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
print("Model ލޯޑްކުރަނީ:", MODEL_ID, "dtype:", model_dtype)
asr_model = haas_asr.load_model(MODEL_ID, device="cuda", dtype=model_dtype)
print("Model ލޯޑްކުރެވިއްޖެ.")


In [ ]:
# @title 6. Transcription
# ============================================================
# 6. Transcription
# ============================================================
import json, traceback
from pathlib import Path
import haas_asr

output_root = Path(OUTPUT_DIR)
output_root.mkdir(parents=True, exist_ok=True)
manifest = []

for source_path in selected_files:
    source_path = Path(source_path)
    print("\n=== Processing ފެށިއްޖެ:", source_path.name, "===")
    item_work = Path(WORK_DIR) / safe_stem(source_path)
    item_work.mkdir(parents=True, exist_ok=True)
    wav_path = item_work / "source_16k_mono.wav"
    record = {"source": source_path.name, "status": "running"}

    try:
        convert_to_mono_16k_wav(source_path, wav_path)
        audio = haas_asr.load_audio(str(wav_path))
        duration = len(audio) / haas_asr.SAMPLE_RATE
        bounds = haas_asr.split_at_quiet_points(
            audio, chunk_seconds=CHUNK_SECONDS,
            search_seconds=2.0,
        )
        if not bounds:
            raise RuntimeError("Haas returned no transcription result.")

        chunk_records = []
        chunk_texts = []
        all_ts = {"char": [], "word": [], "segment": []}
        for idx, (first, last) in enumerate(bounds):
            start = first / haas_asr.SAMPLE_RATE
            end = last / haas_asr.SAMPLE_RATE
            print(f"  chunk {idx+1}/{len(bounds)}: {start:.1f}s - {end:.1f}s")
            # Inference is owned by the official Haas API; no older NeMo
            # TDT/CTC decoder, CUDA-graph patch or timestamp flags survive.
            text = haas_asr.transcribe(
                asr_model, [audio[first:last]],
                batch_size=INFERENCE_BATCH_SIZE,
            )[0]
            chunk_texts.append(text)
            if RETURN_TIMESTAMPS and text.strip():
                all_ts["segment"].append(estimated_segment(text, start, end, idx))
            chunk_records.append({
                "index": idx, "start": start, "end": end,
                "duration": end - start, "text": text,
                "timestamp_counts": {
                    "char": 0, "word": 0,
                    "segment": int(bool(RETURN_TIMESTAMPS and text.strip())),
                },
            })

        full_text = join_text_parts(chunk_texts)
        cues, cue_source = build_subtitle_cues(all_ts)
        payload = {
            "source": source_path.name,
            "model": MODEL_ID,
            "decoder": "TDT",
            "timestamp_mode": TIMESTAMP_MODE,
            "timestamp_precision": "approximate_chunk_bounds" if RETURN_TIMESTAMPS else "none",
            "duration_seconds": duration,
            "chunk_seconds": CHUNK_SECONDS,
            "text": full_text,
            "chunks": chunk_records,
            "timestamps": all_ts,
            "subtitle_source": cue_source,
            "subtitle_cues": cues,
            "diagnostics": {
                "chunk_count": len(bounds),
                "characters": len(full_text),
                "char_timestamps": 0,
                "word_timestamps": 0,
                "segment_timestamps": len(all_ts["segment"]),
                "subtitle_cues": len(cues),
                "subtitle_max_duration": max(
                    (c["end"] - c["start"] for c in cues), default=0.0
                ),
                "subtitle_max_chars": max((len(c["text"]) for c in cues), default=0),
                "timestamps_monotonic": all(
                    rows[i]["start"] >= rows[i - 1]["start"]
                    for rows in all_ts.values() for i in range(1, len(rows))
                ),
            },
        }
        paths = write_outputs(source_path, payload, output_root)
        print("  އަކުރުތައް:", len(full_text))
        print("  native segment timestamps:", 0)
        print("  word timestamps:", 0)
        print("  approximate subtitle cues:", len(cues))
        record.update({
            "status": "ok", "duration_seconds": duration,
            "characters": len(full_text), "chunks": len(bounds),
            "char_timestamps": 0, "word_timestamps": 0,
            "segment_timestamps": len(all_ts["segment"]),
            "subtitle_cues": len(cues), "subtitle_source": cue_source,
            "outputs": [p.name for p in paths],
        })
    except Exception as exc:
        record.update({
            "status": "error", "error_type": type(exc).__name__,
            "error": str(exc), "traceback": traceback.format_exc(),
        })
        print(record["traceback"])
    manifest.append(record)

manifest_path = output_root / "manifest.json"
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("\nProcessing ނިމިއްޖެ.")
print("Manifest:", manifest_path)


In [ ]:
# @title 7. ZIP ސޭވް / download
# ============================================================
# 7. ZIP ސޭވް / download
# ============================================================
from pathlib import Path
import shutil

output_root = Path(OUTPUT_DIR)
zip_base = Path("/content") / Path(ZIP_FILE_NAME).stem
zip_path = Path(shutil.make_archive(str(zip_base), "zip", root_dir=output_root))
print("ZIP:", zip_path)

if SAVE_OUTPUTS_TO_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    drive_out = Path(GOOGLE_DRIVE_ROOT) / "Haas_Japanese_ASR_outputs"
    drive_out.mkdir(parents=True, exist_ok=True)
    target = drive_out / zip_path.name
    shutil.copy2(zip_path, target)
    print("Google Drive ގައި ސޭވްކުރެވިއްޖެ:", target)

if AUTO_DOWNLOAD_ZIP:
    from google.colab import files
    files.download(str(zip_path))
